# 3. QUBO: wybór zadań przy konflikcie

**Cel:** sformułować zadanie jako kwadratową funkcję bitów i znaleźć minimum klasycznie. Czas: 12 min. Wymaga `numpy`.

**Uruchamianie:** wykonuj komórki kolejno. Dane są generowane lokalnie; nie jest potrzebny internet.

### Funkcja energii
Trzy zadania przynoszą wartość 3, 4, 2. Zadania 1 i 2 oraz 2 i 3 wzajemnie się wykluczają. Kara za parę to 6: `E(x)=-3x₁-4x₂-2x₃+6x₁x₂+6x₂x₃`. Zapis QUBO działa na bitach także na zwykłym CPU; sam zapis nie jest obliczeniem kwantowym.

In [ ]:
import numpy as np
from itertools import product
Q=np.array([[-3.,3.,0.],[3.,-4.,3.],[0.,3.,-2.]])
def energy(x):
    x=np.asarray(x,dtype=float)
    return float(x@Q@x)
states=[(np.array(bits,dtype=int),energy(bits)) for bits in product((0,1),repeat=3)]
for bits,value in states: print(''.join(map(str,bits)),f'{value:5.1f}')
best_x,best_e=min(states,key=lambda item:item[1])
print('Optimum:',best_x.tolist(),'E=',best_e)
assert tuple(best_x)==(1,0,1) and best_e==-5.0
assert all(abs(energy(x)-(-3*x[0]-4*x[1]-2*x[2]+6*x[0]*x[1]+6*x[1]*x[2]))<1e-9 for x,_ in states)


### Lokalne przeszukiwanie
Sprawdź, czy losowy start i wybór najlepszej pojedynczej zmiany bitu odnajdują rozwiązanie globalne.

In [ ]:
rng=np.random.default_rng(11)
def descend(x):
    x=np.array(x,dtype=int)
    trail=[energy(x)]
    while True:
        neighbors=[]
        for i in range(3):
            z=x.copy();z[i]=1-z[i];neighbors.append(z)
        candidate=min(neighbors,key=energy)
        if energy(candidate)>=energy(x):break
        x=candidate;trail.append(energy(x))
    return x,trail
for _ in range(4):
    initial=rng.integers(0,2,size=3)
    x,trail=descend(initial)
    print(initial,'->',x,trail)
    assert all(a>=b for a,b in zip(trail,trail[1:]))


**Analiza:** Zmień karę z 6 na 2, przelicz `Q` i sprawdź, kiedy opłaca się wybrać zadania w konflikcie. Złożoność pełnego przeglądu to `2^n`.